In [ ]:
import dask
import matplotlib.pyplot as plt
import xarray as xr

from srm import catalog
from srm.downscaling_utils import interpolate_coarse_to_fine_grid
from srm.qa_flags import DIR_QA_FLAG_CONSTANT_INPUTS

In [ ]:
VARIABLES = ["tas", "tasmax", "tasmin", "pr", "rsds", "hurs"]

# rsds [lat, dayofyear]

Markdown was generated by Claude, then reviewed and edited.
The ceiling is built as a function of `(dayofyear, lat)`: the maximum `rsds` across all longitudes in that latitude band (a "zonal max"), taken across ERA5 and three GCMs (`CESM2-WACCM`, `UKESM`, `MIROC-ES2H`) and three scenario groups (`historical`, `g6_1p5k`, `ssp245`). Because it is the *union* of several models' and scenarios' extremes, further loosened by maxing over longitude, it is deliberately a high bar rather than a tight statistical bound. It only depends on the input data, not on which output is being checked, so it is cached rather than recomputed on every run.

This is an intentionally simpler, standalone check -- it is not connected to the rolling-window plausible-bounds check in `srm.qaqc.calculate_reasonable_bounds_doy`. The evaluation currently covers one debiased output (`CESM2-WACCM`, `ssp245`, ensemble member `003`) as a first pass.

This is of dimensions [lat,dayofyear] and we are calculating a really high max bound: the maximum downwelling solar radiation at the surface across all longitudes in that zonal band, in observations and across historical and future scenarios.

This only needs to be done once. (Only needs to be rerun if the input data changes)

In [ ]:
CACHE_PATH = DIR_QA_FLAG_CONSTANT_INPUTS + "zonal_doy_max_rsds.zarr"


def load_cached(key: str) -> xr.DataArray | None:
    """None on a cache miss; the caller decides what to do about it."""
    try:
        return xr.open_zarr(CACHE_PATH, group=key)["data"].load()
    except FileNotFoundError:
        return None


def save_cached(key: str, da: xr.DataArray) -> None:
    da.rename("data").to_dataset().to_zarr(CACHE_PATH, group=key, mode="w")

Load ERA5 as the fine reference grid: it's both the observational input to the ceiling and the common fine grid every GCM below gets regridded onto, so all sources land on the same grid before combining.

In [ ]:
fine_obs = catalog.get("ERA5").to_xarray()["rsds"]

For one GCM's coarse-resolution time series: reduce to a day-of-year max, regrid that onto the fine ERA5 grid with `interpolate_coarse_to_fine_grid`, then take the max across longitude to get that source's zonal day-of-year bound.

In [ ]:
def calculate_fine_gcm_zonal_limit(coarse_gcm_grid_tseries, fine_obs_doy):
    coarse_gcm_doy_grid = coarse_gcm_grid_tseries.groupby("time.dayofyear").max(dim="time")
    fine_gcm_doy_grid = interpolate_coarse_to_fine_grid(
        da_coarse_to_regrid=coarse_gcm_doy_grid, da_fine_grid=fine_obs_doy
    )
    fine_gcm_doy_zonal = fine_gcm_doy_grid.max(dim="lon")

    return fine_gcm_doy_zonal

The same reduction applied directly to ERA5, which is already on the fine grid so no regridding step is needed.

In [ ]:
fine_obs_doy = fine_obs.groupby("time.dayofyear").max(dim="time")
fine_obs_doy_zonal = fine_obs_doy.max(dim="lon")

Generic cache helpers: write/read a DataArray to/from one zarr store, keyed by a string group name. Nothing here is specific to `rsds`.

Combine every source into one ceiling: for each GCM x scenario combination, compute that source's zonal day-of-year max with the helper above, and keep a running elementwise maximum across all of them, starting from the ERA5 value. `zonal_limit_across_all` is the final ceiling, committed to the cache in the next cell.

In [ ]:
zonal_limit_across_all = fine_obs_doy_zonal

plot_gcm_limits = False

for gcm in ["CESM2-WACCM", "UKESM", "MIROC-ES2H"]:
    print(gcm)
    for scenario in ["historical", "g6_1p5k", "ssp245"]:
        print(scenario)

        gcm_scenario = catalog.get(gcm).to_xarray()[scenario]["rsds"]

        gcm_zonal_limit = (
            calculate_fine_gcm_zonal_limit(gcm_scenario, fine_obs_doy=fine_obs_doy)
            .max(dim="ensemble_member")
            .load()
        )

        zonal_limit_across_all = zonal_limit_across_all.where(
            (zonal_limit_across_all > gcm_zonal_limit) | gcm_zonal_limit.isnull(), gcm_zonal_limit
        ).load()

        if plot_gcm_limits:
            plt.figure()
            gcm_zonal_limit.plot()
            plt.show()

In [ ]:
save_cached(key="zonal_doy_max_rsds", da=zonal_limit_across_all)

# Outlier thresholds from obs

In [ ]:
def calculate_thresholds(obs_max, obs_min, obs_max_std, obs_min_std):
    outlier_thresh_high = obs_max + (5 * obs_max_std)
    outlier_thresh_low = obs_min - (5 * obs_min_std)

    return outlier_thresh_low, outlier_thresh_high

In [ ]:
from srm import catalog

In [ ]:
obs = catalog.get("ERA5").to_xarray()
obs = obs[VARIABLES]

In [ ]:
from srm.downscaling_utils import rechunk

obs = xr.Dataset(
    {var: rechunk(da=obs[var], pattern="full_time") for var in obs.data_vars},
    attrs=obs.attrs,
)

In [ ]:
def calculate_stats(obs=obs, subset=None, timescale="annual"):
    if subset is not None:
        obs_subset = obs.sel(lat=subset[0], lon=subset[1]).load()
    else:
        obs_subset = obs

    if timescale == "annual":
        annual = obs_subset.groupby("time.year")
        obs_annual_max, obs_annual_min = dask.compute(annual.max(), annual.min())

        obs_max = obs_annual_max.max(dim=["year"])
        obs_min = obs_annual_min.min(dim=["year"])

        obs_max_std = obs_annual_max.std(dim=["year"])
        obs_min_std = obs_annual_min.std(dim=["year"])

    elif timescale == "monthly":
        monthly = obs_subset.groupby(["time.year", "time.month"])
        obs_monthly_max, obs_monthly_min = dask.compute(monthly.max(), monthly.min())

        obs_max = obs_monthly_max.max(dim="year")
        obs_min = obs_monthly_min.min(dim="year")
        obs_max_std = obs_monthly_max.std(dim="year")
        obs_min_std = obs_monthly_min.std(dim="year")

    elif timescale == "dayofyear":
        rolling_doy_max = obs_subset.rolling(time=30, center=True).max()
        rolling_doy_min = obs_subset.rolling(time=30, center=True).min()

        obs_max = rolling_doy_max.groupby("time.dayofyear").max()
        obs_min = rolling_doy_min.groupby("time.dayofyear").min()
        obs_max_std = rolling_doy_max.groupby("time.dayofyear").std()
        obs_min_std = rolling_doy_min.groupby("time.dayofyear").std()

    return [obs_max, obs_min, obs_max_std, obs_min_std]

##### Annual stats

In [ ]:
[obs_max, obs_min, obs_max_std, obs_min_std] = calculate_stats(subset=None, timescale="annual")

In [ ]:
[outlier_thresh_low, outlier_thresh_high] = calculate_thresholds(
    obs_max, obs_min, obs_max_std, obs_min_std
)

In [ ]:
combined = xr.merge(
    [
        obs_max.rename({v: f"{v}_max" for v in obs_max.data_vars}),
        obs_min.rename({v: f"{v}_min" for v in obs_min.data_vars}),
        obs_max_std.rename({v: f"{v}_max_std" for v in obs_max_std.data_vars}),
        obs_min_std.rename({v: f"{v}_min_std" for v in obs_min_std.data_vars}),
    ]
)

combined.to_zarr(DIR_QA_FLAG_CONSTANT_INPUTS + "annual_obs_thresholds_global.zarr", mode="w")

##### Day of year stats

In [ ]:
[obs_max, obs_min, obs_max_std, obs_min_std] = calculate_stats(subset=None, timescale="dayofyear")

In [ ]:
# Sometimes this crashes after a few variables -- works if you rerun
STORE = DIR_QA_FLAG_CONSTANT_INPUTS + "doy_obs_thresholds_global.zarr"

stats = {"max": obs_max, "min": obs_min, "max_std": obs_max_std, "min_std": obs_min_std}

for i, var in enumerate(VARIABLES):
    print(var)
    per_var = xr.merge(
        [
            stats["max"][[var]].rename({var: f"{var}_max"}),
            stats["min"][[var]].rename({var: f"{var}_min"}),
            stats["max_std"][[var]].rename({var: f"{var}_max_std"}),
            stats["min_std"][[var]].rename({var: f"{var}_min_std"}),
        ]
    )
    mode = "w" if i == 0 else "a"
    per_var.to_zarr(STORE, mode=mode)